# Part 7: Optical Change Detection

建立双时相光学变化检测最小闭环：配准、Siamese 特征、差异融合、阈值与 F1/IoU。

**先修要求**：完成前序章节，了解 NumPy、张量 shape 与训练/验证划分。

## 学习目标与路线

1. 明确问题与输入输出
2. 理解核心结构
3. 运行最小实验
4. 检查指标和 shape
5. 完成练习与遥感迁移

`image_t1 + image_t2 → shared encoder → feature difference/fusion → change logits → binary mask`

In [ ]:
import numpy as np
print('NumPy:', np.__version__)

## 核心概念

- **Co-registration**：亚像素错位也会制造伪变化，进入模型前先对齐。
- **Siamese encoder**：两个时相共享权重，便于比较一致的特征空间。
- **Change fusion**：可用绝对差、拼接、注意力或双向交互融合。
- **Metrics**：变化像素稀少，应报告 precision、recall、F1、IoU，而非只报 accuracy。

## 最小可运行实验

先用小而确定的合成数据验证关键数学与指标。真实训练时再替换为 PyTorch 模型和 Part 3 的数据流水线。

In [ ]:
import numpy as np
rng = np.random.default_rng(7)
t1 = rng.normal(0.3, 0.03, (64, 64))
t2 = t1.copy(); t2[20:42, 28:50] += 0.45
t2 += rng.normal(0, 0.02, t2.shape)
truth = np.zeros_like(t1, dtype=bool); truth[20:42, 28:50] = True
change_score = np.abs(t2 - t1)
pred = change_score > 0.2
tp = np.logical_and(pred, truth).sum(); fp = np.logical_and(pred, ~truth).sum(); fn = np.logical_and(~pred, truth).sum()
precision = tp / (tp + fp + 1e-8); recall = tp / (tp + fn + 1e-8)
f1 = 2 * precision * recall / (precision + recall + 1e-8)
print({"precision": round(precision, 4), "recall": round(recall, 4), "f1": round(f1, 4)})
assert f1 > 0.95

## 结果解释

检查断言、shape 和指标是否符合预期。若最小实验失败，先修复实现；不要立即增加模型规模或下载更大数据。

## 练习

人为将 t2 平移 1 个像素，比较伪变化数量；再讨论真实项目的配准质量检查。

In [ ]:
# 答案脚手架
experiment = {'hypothesis': '', 'controlled_variables': [], 'metric': '', 'expected_failure': ''}
print(experiment)

## 常见坑与扩展

**常见坑**：时相未配准；季节/光照变化被当成地物变化；正负样本极不平衡；同一区域跨集合泄漏。

**扩展**：替换为真实遥感数据时，记录传感器、波段、分辨率、空间划分和随机种子。

## 完成检查

- [ ] 所有单元格从头运行
- [ ] 指标实现通过 sanity check
- [ ] 没有 train/val 空间泄漏
- [ ] 配置和随机种子已记录
- [ ] 至少分析一个失败案例